# Merge and analyze the parallel main single-shot experiment

This notebook performs no model training.

It first attempts to load the three completed shard snapshots. If
one or more snapshots are unavailable, it reconstructs the same
tables directly from the shared per-trial cache.

Consequently, plots and tables can be regenerated after changing
analysis code without rerunning the scientific experiment.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root containing opinion_dynamics/."
    )


REPO_ROOT = find_repo_root()
print("Repository root:", REPO_ROOT)

In [ ]:
STUDY_DATE = "2026_08_04"
BASE_STUDY_NAME = "single_shot_exploration_sweep_main_parallel"
NUM_PARALLEL_SHARDS = 3
CACHE_VERSION = "v2"
PIPELINE_VERSION = "2026-08-04-parallel-v1"

RESULTS_ROOT = (
    REPO_ROOT
    / "opinion_dynamics"
    / "experiments"
    / "results"
)

SHARD_DIRS = [
    RESULTS_ROOT
    / (
        f"experiment_{STUDY_DATE}_{BASE_STUDY_NAME}"
        f"_shard_{shard_id}_of_{NUM_PARALLEL_SHARDS}"
    )
    for shard_id in range(NUM_PARALLEL_SHARDS)
]

CACHE_DIR = (
    RESULTS_ROOT.parent
    / "_trial_cache"
    / BASE_STUDY_NAME
)

RESULTS_DIR = (
    RESULTS_ROOT
    / f"experiment_{STUDY_DATE}_{BASE_STUDY_NAME}_merged"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DYNAMICS_SPECS = {
    "laplacian": {
        "label": "Linear consensus",
        "env_kwargs": {},
    },
    "coca": {
        "label": "COCA",
        "env_kwargs": {},
    },
    "hegselmannkrause": {
        "label": "Hegselmann--Krause",
        "env_kwargs": {
            "hk_epsilon": 0.50,
            "hk_include_self": True,
        },
    },
    "friedkinjohnsen": {
        "label": "Friedkin--Johnsen",
        "env_kwargs": {
            "fj_lambda": 0.98,
            "fj_prejudice": None,
        },
    },
    "nonlinearinfluence": {
        "label": "Nonlinear influence",
        "env_kwargs": {
            "nonlinear_beta": 4.0,
        },
    },
    "repulsion": {
        "label": "Repulsion",
        "env_kwargs": {
            "repulsion_epsilon": 0.30,
            "repulsion_strength": 0.10,
        },
    },
}

ENABLED_DYNAMICS = list(DYNAMICS_SPECS)
DYNAMICS_LABELS = {
    name: spec["label"]
    for name, spec in DYNAMICS_SPECS.items()
}

TOPOLOGY_SEEDS = [3, 4, 5, 6, 7]
INITIAL_PERMUTATION_SEEDS = [0, 1, 2, 4, 5]
EXPLORATION_CAMPAIGN_COUNTS = list(range(0, 11))

N_AGENTS = 15
OMEGA = 1.0
U_BAR = 0.2
NUM_CAMPAIGNS_TOTAL = 20
T_CAMPAIGN = 0.5
T_S = 0.1
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS_TOTAL - 1)
LEARNED_POLICY_LAMBDA = 0.70

EXPECTED_CONDITION_TRIALS = (
    len(ENABLED_DYNAMICS)
    * len(TOPOLOGY_SEEDS)
    * len(INITIAL_PERMUTATION_SEEDS)
    * len(EXPLORATION_CAMPAIGN_COUNTS)
)
EXPECTED_SUMMARY_ROWS = EXPECTED_CONDITION_TRIALS * 2

SOURCE_MODE = "auto"  # "auto", "shards", or "cache"

print("Merged results directory:", RESULTS_DIR)
print("Shared cache directory:", CACHE_DIR)
print("Expected condition trials:", EXPECTED_CONDITION_TRIALS)
print("Expected learned summary rows:", EXPECTED_SUMMARY_ROWS)

In [ ]:
POLICY_LINEAR = "online_linear_euler"
POLICY_NONLINEAR = "online_nonlinear_lambda_mix"
POLICY_TRUE_GRAPH = "true_graph_centrality"
POLICY_UNIFORM = "uniform"
POLICY_NOCONTROL = "no_control"

POLICY_LABELS = {
    POLICY_LINEAR: "online linear identifier",
    POLICY_NONLINEAR: "online nonlinear identifier",
    POLICY_TRUE_GRAPH: "true-graph centrality",
    POLICY_UNIFORM: "uniform",
    POLICY_NOCONTROL: "no control",
}

LEARNED_POLICIES = [POLICY_LINEAR, POLICY_NONLINEAR]
PLOT_POLICIES = [
    POLICY_TRUE_GRAPH,
    POLICY_LINEAR,
    POLICY_NONLINEAR,
    POLICY_UNIFORM,
    POLICY_NOCONTROL,
]


def rollout_mean_end(out: Dict[str, Any]) -> float:
    return float(np.asarray(out["states"], dtype=float)[-1].mean())


def graph_weighted_target_error(
    states: np.ndarray,
    v_true: np.ndarray,
    omega: float = OMEGA,
) -> float:
    """Secondary diagnostic; not a consensus error for nonlinear models."""
    final_state = np.asarray(states, dtype=float)[-1]
    graph_weighted_value = float(
        np.asarray(v_true, dtype=float).reshape(-1) @ final_state
    )
    return abs(float(omega) - graph_weighted_value)


def trajectory_rows(
    *,
    dynamics: str,
    topology_seed: int,
    initial_seed: int,
    policy: str,
    rollout: Dict[str, Any],
    v_true: np.ndarray,
) -> List[Dict[str, Any]]:
    rows = []
    states = np.asarray(rollout["states"], dtype=float)
    times = np.asarray(rollout["boundary_times"], dtype=float)
    trial_id = f"{dynamics}|topo={topology_seed}|init={initial_seed}"
    for idx, (t, x) in enumerate(zip(times, states)):
        rows.append(
            {
                "dynamics": dynamics,
                "dynamics_label": DYNAMICS_LABELS[dynamics],
                "topology_seed": int(topology_seed),
                "initial_seed": int(initial_seed),
                "trial_id": trial_id,
                "policy": policy,
                "policy_label": POLICY_LABELS.get(policy, policy),
                "boundary_index": int(idx),
                "time": float(t),
                "mean_opinion": float(np.mean(x)),
                "min_opinion": float(np.min(x)),
                "max_opinion": float(np.max(x)),
                "graph_weighted_opinion": float(
                    np.asarray(v_true).reshape(-1)
                    @ np.asarray(x).reshape(-1)
                ),
            }
        )
    return rows


def learned_identifier_metrics(
    *,
    model_name: str,
    learned: Dict[str, Any],
    A_true: np.ndarray,
    v_true: np.ndarray,
) -> Dict[str, Any]:
    A_hats = learned.get("A_hats", [])
    if model_name == POLICY_NONLINEAR:
        v_hats = (
            learned.get("v_hats_lambda", [])
            or learned.get("v_hats_static", [])
        )
    else:
        v_hats = learned.get("v_hats", [])

    if not A_hats or not v_hats:
        return {}

    A_final = np.asarray(A_hats[-1], dtype=float)
    v_final = np.asarray(v_hats[-1], dtype=float)
    v_errs = [
        float(
            np.sum(
                np.abs(np.asarray(vh, dtype=float) - v_true)
            )
        )
        for vh in v_hats
    ]
    last_fit = learned["fit_infos"][-1]
    return {
        "max_v_L1": float(max(v_errs)),
        "final_v_L1": float(np.sum(np.abs(v_final - v_true))),
        "A_MAE_final": float(np.mean(np.abs(A_final - A_true))),
        "A_Fro_final": float(
            np.linalg.norm(A_final - A_true, ord="fro")
        ),
        "final_train_mae": float(last_fit["train_mae"]),
        "final_identity_mae": float(last_fit["identity_mae"]),
        "final_model_over_identity": float(
            last_fit["model_over_identity"]
        ),
        "final_n_pairs": int(last_fit["n_pairs"]),
        "total_fit_time_sec": float(
            sum(
                info.get(
                    "fit_time_sec",
                    info.get("fit_elapsed_s", 0.0),
                )
                for info in learned["fit_infos"]
            )
        ),
    }


def summarize_learned_trial(
    *,
    dynamics: str,
    topology_seed: int,
    initial_seed: int,
    model_name: str,
    learned: Dict[str, Any],
    true_graph: Dict[str, Any],
    uniform: Dict[str, Any],
    no_control: Dict[str, Any],
    A_true: np.ndarray,
    v_true: np.ndarray,
) -> Dict[str, Any]:
    mean_end = rollout_mean_end(learned)
    mean_true_graph = rollout_mean_end(true_graph)
    mean_uniform = rollout_mean_end(uniform)
    mean_nocontrol = rollout_mean_end(no_control)

    row = {
        "dynamics": dynamics,
        "dynamics_label": DYNAMICS_LABELS[dynamics],
        "topology_seed": int(topology_seed),
        "initial_seed": int(initial_seed),
        "trial_id": (
            f"{dynamics}|topo={topology_seed}|init={initial_seed}"
        ),
        "model": model_name,
        "model_label": POLICY_LABELS[model_name],
        "mean_end": float(mean_end),
        "mean_true_graph_end": float(mean_true_graph),
        "mean_uniform_end": float(mean_uniform),
        "mean_nocontrol_end": float(mean_nocontrol),
        "model_minus_true_graph_mean_end": float(
            mean_end - mean_true_graph
        ),
        "model_minus_uniform_mean_end": float(
            mean_end - mean_uniform
        ),
        "model_minus_nocontrol_mean_end": float(
            mean_end - mean_nocontrol
        ),
        "graph_weighted_target_error": graph_weighted_target_error(
            learned["states"],
            v_true,
        ),
    }
    row.update(
        learned_identifier_metrics(
            model_name=model_name,
            learned=learned,
            A_true=A_true,
            v_true=v_true,
        )
    )
    return row


def fit_rows_from_rollout(
    *,
    dynamics: str,
    topology_seed: int,
    initial_seed: int,
    model_name: str,
    learned: Dict[str, Any],
    v_true: np.ndarray,
) -> List[Dict[str, Any]]:
    rows = []
    if model_name == POLICY_NONLINEAR:
        v_hats = (
            learned.get("v_hats_lambda", [])
            or learned.get("v_hats_static", [])
        )
    else:
        v_hats = learned.get("v_hats", [])

    for idx, info in enumerate(learned.get("fit_infos", [])):
        row = {
            "dynamics": dynamics,
            "dynamics_label": DYNAMICS_LABELS[dynamics],
            "topology_seed": int(topology_seed),
            "initial_seed": int(initial_seed),
            "trial_id": (
                f"{dynamics}|topo={topology_seed}|init={initial_seed}"
            ),
            "model": model_name,
            "model_label": POLICY_LABELS[model_name],
        }
        row.update(info)
        if idx < len(v_hats):
            row["v_L1_to_true"] = float(
                np.sum(
                    np.abs(
                        np.asarray(v_hats[idx], dtype=float)
                        - v_true
                    )
                )
            )
        rows.append(row)
    return rows


In [ ]:
SHARD_FILENAMES = {
    "summary": "exploration_sweep_summary.csv",
    "trajectory": "exploration_sweep_trajectories.csv",
    "fit": "exploration_sweep_fit_diagnostics.csv",
    "failed": "exploration_sweep_failed_trials.csv",
    "run_index": "exploration_sweep_run_index.csv",
    "trial_settings": "exploration_sweep_trial_settings.csv",
}


def read_csv_if_nonempty(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def shard_snapshots_are_complete() -> bool:
    for directory in SHARD_DIRS:
        marker = directory / "SHARD_COMPLETE.json"
        if not marker.exists():
            return False

        try:
            payload = json.loads(marker.read_text(encoding="utf-8"))
        except Exception:
            return False

        if payload.get("status") != "success":
            return False
        if payload.get("base_study_name") != BASE_STUDY_NAME:
            return False
        if payload.get("cache_version") != CACHE_VERSION:
            return False
        if payload.get("pipeline_version") != PIPELINE_VERSION:
            return False

        for filename in SHARD_FILENAMES.values():
            if not (directory / filename).exists():
                return False

    return True


def load_from_shard_snapshots() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    summary = pd.concat(
        [
            read_csv_if_nonempty(
                directory / SHARD_FILENAMES["summary"]
            )
            for directory in SHARD_DIRS
        ],
        ignore_index=True,
    )
    trajectory = pd.concat(
        [
            read_csv_if_nonempty(
                directory / SHARD_FILENAMES["trajectory"]
            )
            for directory in SHARD_DIRS
        ],
        ignore_index=True,
    )
    fit = pd.concat(
        [
            read_csv_if_nonempty(
                directory / SHARD_FILENAMES["fit"]
            )
            for directory in SHARD_DIRS
        ],
        ignore_index=True,
    )

    failed_frames = [
        read_csv_if_nonempty(
            directory / SHARD_FILENAMES["failed"]
        )
        for directory in SHARD_DIRS
    ]
    failed_frames = [
        frame for frame in failed_frames if not frame.empty
    ]
    failed = (
        pd.concat(failed_frames, ignore_index=True)
        if failed_frames
        else pd.DataFrame()
    )

    run_index = pd.concat(
        [
            read_csv_if_nonempty(
                directory / SHARD_FILENAMES["run_index"]
            )
            for directory in SHARD_DIRS
        ],
        ignore_index=True,
    )
    trial_settings = pd.concat(
        [
            read_csv_if_nonempty(
                directory / SHARD_FILENAMES["trial_settings"]
            )
            for directory in SHARD_DIRS
        ],
        ignore_index=True,
    )

    return (
        summary,
        trajectory,
        fit,
        failed,
        run_index,
        trial_settings,
    )


def read_json_records(path: Path) -> list[dict[str, Any]]:
    payload = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        return [payload]

    raise TypeError(f"Unexpected JSON payload in {path}")


def load_from_shared_cache() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    if not CACHE_DIR.exists():
        raise FileNotFoundError(CACHE_DIR)

    summary_records: list[dict[str, Any]] = []
    trajectory_records: list[dict[str, Any]] = []
    fit_records: list[dict[str, Any]] = []
    run_index_records: list[dict[str, Any]] = []
    trial_settings_records: list[dict[str, Any]] = []

    accepted = 0
    ignored = 0

    for directory in sorted(CACHE_DIR.iterdir()):
        if not directory.is_dir() or directory.name.startswith("."):
            continue
        if directory.name == "_failures":
            continue

        manifest_path = directory / "manifest.json"
        summary_path = directory / "summary.json"
        trajectory_path = directory / "trajectory.json"
        fit_path = directory / "fit.json"

        if not all(
            path.exists()
            for path in [
                manifest_path,
                summary_path,
                trajectory_path,
                fit_path,
            ]
        ):
            ignored += 1
            continue

        try:
            manifest = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            settings = manifest.get("settings", {})
        except Exception:
            ignored += 1
            continue

        if manifest.get("status") != "success":
            ignored += 1
            continue
        if settings.get("study_name") != BASE_STUDY_NAME:
            ignored += 1
            continue
        if settings.get("cache_version") != CACHE_VERSION:
            ignored += 1
            continue
        if settings.get("pipeline_version") != PIPELINE_VERSION:
            ignored += 1
            continue

        summary_records.extend(read_json_records(summary_path))
        trajectory_records.extend(
            read_json_records(trajectory_path)
        )
        fit_records.extend(read_json_records(fit_path))

        trial_hash = manifest.get(
            "trial_hash",
            directory.name,
        )
        run_index_records.append(
            {
                "trial_hash": trial_hash,
                "dynamics": settings.get("dynamics"),
                "topology_seed": settings.get("topology_seed"),
                "initial_seed": settings.get("initial_seed"),
                "condition": settings.get("condition"),
                "exploration_campaigns": settings.get(
                    "exploration_campaigns"
                ),
                "status": "cache_reconstructed",
                "cache_path": str(directory),
            }
        )
        trial_settings_records.append(
            {
                "trial_hash": trial_hash,
                **settings,
            }
        )
        accepted += 1

    failure_records: list[dict[str, Any]] = []
    failure_dir = CACHE_DIR / "_failures"
    if failure_dir.exists():
        for path in sorted(failure_dir.glob("*.json")):
            try:
                failure_records.append(
                    json.loads(path.read_text(encoding="utf-8"))
                )
            except Exception:
                pass

    print("Accepted cache entries:", accepted)
    print("Ignored cache entries:", ignored)

    return (
        pd.DataFrame(summary_records),
        pd.DataFrame(trajectory_records),
        pd.DataFrame(fit_records),
        pd.DataFrame(failure_records),
        pd.DataFrame(run_index_records),
        pd.json_normalize(trial_settings_records),
    )


if SOURCE_MODE not in {"auto", "shards", "cache"}:
    raise ValueError(SOURCE_MODE)

use_shards = (
    SOURCE_MODE == "shards"
    or (
        SOURCE_MODE == "auto"
        and shard_snapshots_are_complete()
    )
)

if use_shards:
    print("Loading completed shard snapshots.")
    (
        summary_df,
        trajectory_df,
        fit_df,
        failed_df,
        run_index_df,
        trial_settings_df,
    ) = load_from_shard_snapshots()
    DATA_SOURCE = "shard_snapshots"
else:
    print("Loading directly from the shared per-trial cache.")
    (
        summary_df,
        trajectory_df,
        fit_df,
        failed_df,
        run_index_df,
        trial_settings_df,
    ) = load_from_shared_cache()
    DATA_SOURCE = "shared_cache"

print("Data source:", DATA_SOURCE)
print("Summary rows:", len(summary_df))
print("Trajectory rows:", len(trajectory_df))
print("Fit rows:", len(fit_df))
print("Failure rows:", len(failed_df))
print("Trial settings rows:", len(trial_settings_df))

In [ ]:
summary_key = [
    "dynamics",
    "topology_seed",
    "initial_seed",
    "model",
    "exploration_campaigns_config",
]

duplicate_summary = summary_df.duplicated(
    subset=summary_key,
    keep=False,
)

if duplicate_summary.any():
    display(
        summary_df.loc[duplicate_summary]
        .sort_values(summary_key)
        .head(100)
    )
    raise RuntimeError(
        "Duplicate scientific summary keys were found."
    )

if len(summary_df) != EXPECTED_SUMMARY_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_SUMMARY_ROWS} learned summary rows, "
        f"found {len(summary_df)}."
    )

observed_dynamics = set(summary_df["dynamics"].unique())
expected_dynamics = set(ENABLED_DYNAMICS)
if observed_dynamics != expected_dynamics:
    raise RuntimeError(
        f"Dynamics mismatch. Expected {expected_dynamics}, "
        f"found {observed_dynamics}."
    )

observed_k = set(
    summary_df["exploration_campaigns_config"]
    .astype(int)
    .unique()
)
if observed_k != set(EXPLORATION_CAMPAIGN_COUNTS):
    raise RuntimeError(
        f"Exploration-count mismatch: {observed_k}"
    )

if not failed_df.empty:
    display(failed_df.head(100))
    raise RuntimeError(
        "At least one failed trial is present."
    )

coverage = (
    summary_df.groupby(
        [
            "dynamics",
            "model",
            "exploration_campaigns_config",
        ]
    )
    .size()
    .rename("trials")
    .reset_index()
)

expected_repetitions = (
    len(TOPOLOGY_SEEDS)
    * len(INITIAL_PERMUTATION_SEEDS)
)
if not np.all(
    coverage["trials"].to_numpy()
    == expected_repetitions
):
    display(coverage)
    raise RuntimeError(
        "At least one dynamics/model/k cell has incomplete coverage."
    )

print("Validation passed.")
display(coverage.head(20))

In [ ]:
summary_df.to_csv(
    RESULTS_DIR / "exploration_sweep_summary_merged.csv",
    index=False,
)
trajectory_df.to_csv(
    RESULTS_DIR / "exploration_sweep_trajectories_merged.csv",
    index=False,
)
fit_df.to_csv(
    RESULTS_DIR / "exploration_sweep_fit_diagnostics_merged.csv",
    index=False,
)
run_index_df.to_csv(
    RESULTS_DIR / "exploration_sweep_run_index_merged.csv",
    index=False,
)
trial_settings_df.to_csv(
    RESULTS_DIR / "exploration_sweep_trial_settings_merged.csv",
    index=False,
)

merge_manifest = {
    "status": "success",
    "data_source": DATA_SOURCE,
    "study_date": STUDY_DATE,
    "base_study_name": BASE_STUDY_NAME,
    "cache_version": CACHE_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "expected_summary_rows": EXPECTED_SUMMARY_ROWS,
    "summary_rows": int(len(summary_df)),
    "trajectory_rows": int(len(trajectory_df)),
    "fit_rows": int(len(fit_df)),
}

(
    RESULTS_DIR / "MERGE_COMPLETE.json"
).write_text(
    json.dumps(
        merge_manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print("Saved merged raw tables to:", RESULTS_DIR)

# Results at a glance

**Start here.** The primary score is the final average-opinion gap over the uniform-allocation baseline. A positive value means the learned policy outperformed uniform allocation on the same trial.

The so-called oracle from the earlier notebook is renamed **true-graph centrality**. For nonlinear propagation models it is a graph-informed benchmark, not necessarily a globally optimal controller.


In [ ]:
def mean_ci95(
    df: pd.DataFrame,
    value_col: str,
    group_cols: list[str],
) -> pd.DataFrame:
    out = (
        df.groupby(group_cols)[value_col]
        .agg(mean="mean", std="std", count="count")
        .reset_index()
    )
    out["sem"] = (
        out["std"]
        / np.sqrt(out["count"].clip(lower=1))
    )
    out["ci95"] = 1.96 * out["sem"].fillna(0.0)
    return out


decision_summary = mean_ci95(
    summary_df,
    "model_minus_uniform_mean_end",
    [
        "dynamics",
        "dynamics_label",
        "model",
        "model_label",
        "exploration_campaigns_config",
    ],
)

win_rates = (
    summary_df.assign(
        win_vs_uniform=(
            summary_df["model_minus_uniform_mean_end"] > 0
        )
    )
    .groupby(
        [
            "dynamics",
            "model",
            "exploration_campaigns_config",
        ]
    )["win_vs_uniform"]
    .mean()
    .rename("win_rate_vs_uniform")
    .reset_index()
)
decision_summary = decision_summary.merge(
    win_rates,
    on=[
        "dynamics",
        "model",
        "exploration_campaigns_config",
    ],
    how="left",
)

best_rows = (
    decision_summary.sort_values(
        [
            "dynamics",
            "model",
            "mean",
            "exploration_campaigns_config",
        ],
        ascending=[True, True, False, True],
    )
    .groupby(["dynamics", "model"], as_index=False)
    .head(1)
    .copy()
)

best_table = best_rows[
    [
        "dynamics_label",
        "model_label",
        "exploration_campaigns_config",
        "mean",
        "ci95",
        "win_rate_vs_uniform",
        "count",
    ]
].rename(
    columns={
        "exploration_campaigns_config": "best_exploration_campaigns",
        "mean": "best_mean_gap_vs_uniform",
        "ci95": "best_gap_ci95",
        "count": "n_trials",
    }
)

print("Primary result: best exploration count for each propagation model and identifier")
display(
    best_table.sort_values(
        ["dynamics_label", "model_label"]
    ).round(5)
)

robust_summary = (
    best_table.groupby("model_label")[
        "best_exploration_campaigns"
    ]
    .agg(
        median_best_k="median",
        min_best_k="min",
        max_best_k="max",
    )
    .reset_index()
)
print("Across-propagation robustness summary")
display(robust_summary)


## 1. What does exploration add relative to no exploration?

Each cell reports the paired change in final average opinion relative to \(k=0\), using the same propagation model, topology, initial state, and identifier. The star marks the best observed count in each row.


In [ ]:
# Pair every k with k=0 using the same dynamics, topology, initial state,
# and identifier. This is easier to interpret than raw final values:
# positive means that k exploration campaigns improved over no exploration.
pair_keys = [
    "dynamics",
    "dynamics_label",
    "topology_seed",
    "initial_seed",
    "trial_id",
    "model",
    "model_label",
]

zero = (
    summary_df[
        summary_df["exploration_campaigns_config"] == 0
    ][pair_keys + ["mean_end"]]
    .rename(columns={"mean_end": "mean_end_k0"})
)

paired_effect_df = summary_df.merge(
    zero,
    on=pair_keys,
    how="inner",
)
paired_effect_df["gain_vs_k0"] = (
    paired_effect_df["mean_end"]
    - paired_effect_df["mean_end_k0"]
)

paired_effect_summary = mean_ci95(
    paired_effect_df,
    "gain_vs_k0",
    [
        "dynamics",
        "dynamics_label",
        "model",
        "model_label",
        "exploration_campaigns_config",
    ],
)

def annotated_heatmap(
    table: pd.DataFrame,
    *,
    title: str,
    value_label: str,
    center_zero: bool = True,
) -> None:
    values = table.to_numpy(dtype=float)
    fig, ax = plt.subplots(
        figsize=(max(8, 0.72 * len(table.columns) + 3), 4.5)
    )
    if center_zero:
        vmax = float(np.nanmax(np.abs(values))) if values.size else 1.0
        vmax = max(vmax, 1e-12)
        im = ax.imshow(
            values,
            aspect="auto",
            vmin=-vmax,
            vmax=vmax,
        )
    else:
        im = ax.imshow(values, aspect="auto")

    ax.set_title(title)
    ax.set_xlabel("number of exploration campaigns")
    ax.set_ylabel("propagation model")
    ax.set_xticks(np.arange(len(table.columns)))
    ax.set_xticklabels(table.columns)
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels(table.index)

    for i in range(table.shape[0]):
        row = values[i]
        finite = np.isfinite(row)
        best_j = (
            int(np.nanargmax(row))
            if finite.any()
            else None
        )
        for j in range(table.shape[1]):
            val = values[i, j]
            if not np.isfinite(val):
                text = ""
            else:
                text = f"{val:+.3f}"
                if best_j == j:
                    text += "\n★"
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=9,
            )

    fig.colorbar(im, ax=ax, label=value_label)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


for model in LEARNED_POLICIES:
    sub = paired_effect_summary[
        paired_effect_summary["model"] == model
    ]
    table = (
        sub.pivot(
            index="dynamics_label",
            columns="exploration_campaigns_config",
            values="mean",
        )
        .reindex(
            index=[
                DYNAMICS_LABELS[d]
                for d in ENABLED_DYNAMICS
            ],
            columns=EXPLORATION_CAMPAIGN_COUNTS,
        )
    )
    annotated_heatmap(
        table,
        title=(
            f"{POLICY_LABELS[model]}: paired gain over "
            "0 exploration campaigns"
        ),
        value_label="final average-opinion gain",
        center_zero=True,
    )

display(
    paired_effect_summary.sort_values(
        [
            "dynamics_label",
            "model_label",
            "exploration_campaigns_config",
        ]
    ).round(5)
)


## 2. Which exploration count is selected?

This plot isolates the final decision of interest instead of mixing it with fit diagnostics and trajectory plots.


In [ ]:
# Visual summary of the selected k for each propagation model.
fig, ax = plt.subplots(figsize=(9, 5))
y_labels = [
    DYNAMICS_LABELS[d]
    for d in ENABLED_DYNAMICS
]
y_pos = np.arange(len(y_labels), dtype=float)
offsets = {
    POLICY_LINEAR: -0.12,
    POLICY_NONLINEAR: 0.12,
}
markers = {
    POLICY_LINEAR: "o",
    POLICY_NONLINEAR: "s",
}

for model in LEARNED_POLICIES:
    sub = best_rows[
        best_rows["model"] == model
    ].set_index("dynamics_label")
    x = [
        float(
            sub.loc[label, "exploration_campaigns_config"]
        )
        for label in y_labels
    ]
    ax.scatter(
        x,
        y_pos + offsets[model],
        marker=markers[model],
        s=90,
        label=POLICY_LABELS[model],
    )
    for xi, yi in zip(x, y_pos + offsets[model]):
        ax.text(
            xi + 0.12,
            yi,
            f"k={int(xi)}",
            va="center",
            fontsize=9,
        )

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels)
ax.set_xticks(EXPLORATION_CAMPAIGN_COUNTS)
ax.set_xlim(
    min(EXPLORATION_CAMPAIGN_COUNTS) - 0.5,
    max(EXPLORATION_CAMPAIGN_COUNTS) + 0.8,
)
ax.set_xlabel("selected number of exploration campaigns")
ax.set_title(
    "Best exploration count depends on propagation model"
)
ax.grid(True, axis="x", alpha=0.3)
ax.legend(loc="best")
plt.tight_layout()
plt.show()
plt.close(fig)


## 3. How costly is choosing the wrong count?

These heatmaps show performance relative to the best observed \(k\) within each propagation-model/identifier pair. Values near zero indicate a broad, forgiving plateau; large negative values indicate that the choice of \(k\) matters.


In [ ]:
# Regret is expressed relative to the best observed k within each
# propagation-model/identifier combination. The best cell is 0; more
# negative cells indicate a more consequential choice of k.
best_means = (
    decision_summary.groupby(
        ["dynamics", "model"]
    )["mean"]
    .max()
    .rename("best_mean")
    .reset_index()
)
selection_sensitivity = decision_summary.merge(
    best_means,
    on=["dynamics", "model"],
    how="left",
)
selection_sensitivity["regret_vs_best_k"] = (
    selection_sensitivity["mean"]
    - selection_sensitivity["best_mean"]
)

for model in LEARNED_POLICIES:
    sub = selection_sensitivity[
        selection_sensitivity["model"] == model
    ]
    table = (
        sub.pivot(
            index="dynamics_label",
            columns="exploration_campaigns_config",
            values="regret_vs_best_k",
        )
        .reindex(
            index=[
                DYNAMICS_LABELS[d]
                for d in ENABLED_DYNAMICS
            ],
            columns=EXPLORATION_CAMPAIGN_COUNTS,
        )
    )
    annotated_heatmap(
        table,
        title=(
            f"{POLICY_LABELS[model]}: loss relative "
            "to the best observed exploration count"
        ),
        value_label="final average-opinion difference",
        center_zero=False,
    )

selection_sensitivity.to_csv(
    RESULTS_DIR / "exploration_selection_sensitivity.csv",
    index=False,
)


## 4. Raw exploration curves by propagation model

These are the conventional mean-with-95%-CI curves. They are useful for seeing the overall shape, but the paired and sensitivity plots above should be easier to interpret for model selection.


In [ ]:
# Raw gap over uniform. One figure per identifier keeps the main comparison
# readable while showing propagation-model dependence.
for model in LEARNED_POLICIES:
    fig, ax = plt.subplots(figsize=(9, 5))
    sub_model = decision_summary[
        decision_summary["model"] == model
    ]
    for dynamics in ENABLED_DYNAMICS:
        sub = sub_model[
            sub_model["dynamics"] == dynamics
        ].sort_values("exploration_campaigns_config")
        x = sub["exploration_campaigns_config"].to_numpy(
            dtype=float
        )
        y = sub["mean"].to_numpy(dtype=float)
        e = sub["ci95"].to_numpy(dtype=float)
        ax.plot(
            x,
            y,
            marker="o",
            label=DYNAMICS_LABELS[dynamics],
        )
        ax.fill_between(
            x,
            y - e,
            y + e,
            alpha=0.12,
        )

    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("number of exploration campaigns")
    ax.set_ylabel("final average-opinion gap over uniform")
    ax.set_title(
        f"{POLICY_LABELS[model]}: exploration sweep "
        "by propagation model"
    )
    ax.set_xticks(EXPLORATION_CAMPAIGN_COUNTS)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# Optional diagnostics

Everything below is secondary to the decision plots above. Use these sections to understand *why* a particular exploration count performed well or poorly.


In [ ]:
# Identifier fit quality is a secondary diagnostic. It becomes especially
# important when the true propagation law is nonlinear and the linear
# identifier is deliberately misspecified.
if not summary_df.empty:
    fit_summary = mean_ci95(
        summary_df.dropna(
            subset=["final_model_over_identity"]
        ),
        "final_model_over_identity",
        [
            "dynamics",
            "dynamics_label",
            "model",
            "model_label",
            "exploration_campaigns_config",
        ],
    )

    for model in LEARNED_POLICIES:
        fig, ax = plt.subplots(figsize=(9, 5))
        sub_model = fit_summary[
            fit_summary["model"] == model
        ]
        for dynamics in ENABLED_DYNAMICS:
            sub = sub_model[
                sub_model["dynamics"] == dynamics
            ].sort_values("exploration_campaigns_config")
            ax.plot(
                sub["exploration_campaigns_config"],
                sub["mean"],
                marker="o",
                label=DYNAMICS_LABELS[dynamics],
            )
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_xlabel("number of exploration campaigns")
        ax.set_ylabel("model MAE / identity MAE")
        ax.set_title(
            f"{POLICY_LABELS[model]}: fit quality "
            "by propagation model"
        )
        ax.set_xticks(EXPLORATION_CAMPAIGN_COUNTS)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


## Best and worst average-opinion trajectories

For each propagation model and identifier, only the exploration counts with the
best and worst mean final performance are shown. Figures are closed immediately
after display to avoid accumulating Matplotlib objects in memory.


In [ ]:

selection_rows: List[Dict[str, Any]] = []

for dynamics in ENABLED_DYNAMICS:
    model_summaries = {}
    for model in LEARNED_POLICIES:
        sub = decision_summary[
            (decision_summary["dynamics"] == dynamics)
            & (decision_summary["model"] == model)
        ].sort_values(
            ["mean", "exploration_campaigns_config"],
            ascending=[False, True],
        )
        if sub.empty:
            continue
        best_row = sub.iloc[0]
        worst_row = sub.sort_values(
            ["mean", "exploration_campaigns_config"],
            ascending=[True, True],
        ).iloc[0]
        model_summaries[model] = {
            "best_k": int(best_row["exploration_campaigns_config"]),
            "worst_k": int(worst_row["exploration_campaigns_config"]),
            "best_mean_gap": float(best_row["mean"]),
            "worst_mean_gap": float(worst_row["mean"]),
        }
        selection_rows.append(
            {
                "dynamics": dynamics,
                "dynamics_label": DYNAMICS_LABELS[dynamics],
                "model": model,
                "model_label": POLICY_LABELS[model],
                **model_summaries[model],
            }
        )

    if not model_summaries:
        continue

    fig, axes = plt.subplots(
        1,
        len(model_summaries),
        figsize=(7 * len(model_summaries), 5),
        squeeze=False,
    )

    for ax, (model, selected) in zip(axes[0], model_summaries.items()):
        best_k = selected["best_k"]
        worst_k = selected["worst_k"]

        for exploration_count, curve_label, line_style in [
            (best_k, f"best k={best_k}", "-"),
            (worst_k, f"worst k={worst_k}", "--"),
        ]:
            sub = trajectory_df[
                (trajectory_df["dynamics"] == dynamics)
                & (trajectory_df["policy"] == model)
                & (
                    trajectory_df["exploration_campaigns_config"]
                    == exploration_count
                )
            ]
            if sub.empty:
                continue

            grouped = sub.groupby("time")["mean_opinion"]
            mean = grouped.mean()
            count = grouped.count()
            sem = (
                grouped.std(ddof=1)
                / np.sqrt(count.clip(lower=1))
            ).fillna(0.0)

            x = mean.index.to_numpy(dtype=float)
            y = mean.to_numpy(dtype=float)
            e = 1.96 * sem.to_numpy(dtype=float)

            ax.plot(
                x,
                y,
                linestyle=line_style,
                linewidth=2,
                marker="o",
                markersize=3,
                label=curve_label,
            )
            ax.fill_between(
                x,
                y - e,
                y + e,
                alpha=0.10,
            )

        # Uniform is independent of k. Select one condition to avoid plotting
        # duplicate baseline rows stored for every exploration count.
        uniform_sub = trajectory_df[
            (trajectory_df["dynamics"] == dynamics)
            & (trajectory_df["policy"] == POLICY_UNIFORM)
            & (
                trajectory_df["exploration_campaigns_config"]
                == best_k
            )
        ]
        if not uniform_sub.empty:
            uniform_mean = (
                uniform_sub.groupby("time")["mean_opinion"]
                .mean()
            )
            ax.plot(
                uniform_mean.index.to_numpy(dtype=float),
                uniform_mean.to_numpy(dtype=float),
                linestyle=":",
                linewidth=2,
                label="uniform baseline",
            )

        ax.set_title(POLICY_LABELS[model])
        ax.set_xlabel("time")
        ax.set_ylabel("average network opinion")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")

    fig.suptitle(
        f"{DYNAMICS_LABELS[dynamics]}: best versus worst exploration length"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

best_worst_trajectory_table = pd.DataFrame(selection_rows)
best_worst_trajectory_table.to_csv(
    RESULTS_DIR / "best_worst_trajectory_selection.csv",
    index=False,
)
display(best_worst_trajectory_table)


## Extending the propagation-model sweep

To add a propagation model, enable its entry in `DYNAMICS_SPECS`. The run loop, tables, and plots will include it automatically.

Important interpretation notes:

- `laplacian` and `degroot` are aliases in the current environment, so they should not both be included.
- The linear identifier is correctly specified only for linear consensus. Under COCA, HK, FJ, nonlinear influence, or repulsion, it is intentionally a misspecified baseline.
- Graph-centrality recovery and graph-weighted target error are secondary diagnostics for nonlinear models; final average opinion and paired performance against uniform remain the primary metrics.
- Repulsion can fail to converge or produce qualitatively different behavior, so enable it as a separate robustness study rather than silently mixing it into the main paper result.
